## Import Library

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from itertools import product
import os
import re

## Konfigurasi Awal

In [2]:
DATA_DIR = "/content/data"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RUN_TIME = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

LOG_RECORDS = []
NOTES = []

## Mapping Bulan, Komoditas, dan Region

In [3]:
MONTH_MAP = {
    'januari': 1, 'februari': 2, 'maret': 3, 'april': 4,
    'mei': 5, 'juni': 6, 'juli': 7, 'agustus': 8,
    'september': 9, 'oktober': 10, 'november': 11, 'desember': 12,
    'january': 1, 'february': 2, 'march': 3, 'april': 4,
    'may': 5, 'june': 6, 'july': 7, 'august': 8,
    'september': 9, 'october': 10, 'november': 11, 'december': 12,
}

COMMODITY_MAP = {
    'beras setra i/premium': 'beras_setra_premium',
    'beras setra i / premium': 'beras_setra_premium',
    'beras setra i/premium (kg)': 'beras_setra_premium',
    'cabe merah keriting': 'cabai_merah_keriting',
    'cabai merah keriting': 'cabai_merah_keriting',
    'cabe merah keriting (kg)': 'cabai_merah_keriting',
    'cabe rawit merah': 'cabai_rawit_merah',
    'cabai rawit merah': 'cabai_rawit_merah',
    'cabe rawit merah (kg)': 'cabai_rawit_merah',
    'bawang merah': 'bawang_merah',
    'bawang merah (kg)': 'bawang_merah',
    'bawang putih': 'bawang_putih',
    'bawang putih (kg)': 'bawang_putih',
    'daging sapi murni (semur)': 'daging_sapi_murni',
    'daging sapi murni': 'daging_sapi_murni',
    'daging sapi murni (semur) (kg)': 'daging_sapi_murni',
    'ayam broiler/ras': 'ayam_ras',
    'ayam ras': 'ayam_ras',
    'ayam broiler/ras (kg)': 'ayam_ras',
    'telur ayam ras': 'telur_ayam_ras',
    'telur ayam ras (butir)': 'telur_ayam_ras',
    'telur ayam ras (kg)': 'telur_ayam_ras',
    'gula pasir': 'gula_pasir',
    'gula pasir (kg)': 'gula_pasir',
    'beras medium': 'beras_medium',
    'beras medium (kg)': 'beras_medium',
}

TARGET_COMMODITIES = sorted(set(COMMODITY_MAP.values()))

REGION_MAP = {
    'barat': 'Jakarta Barat',
    'selatan': 'Jakarta Selatan',
    'timur': 'Jakarta Timur',
    'utara': 'Jakarta Utara',
    'pusat': 'Jakarta Pusat',
}

REGIONS = sorted(REGION_MAP.values())

print('Target komoditas:', TARGET_COMMODITIES)
print('Regions:', REGIONS)

Target komoditas: ['ayam_ras', 'bawang_merah', 'bawang_putih', 'beras_medium', 'beras_setra_premium', 'cabai_merah_keriting', 'cabai_rawit_merah', 'daging_sapi_murni', 'gula_pasir', 'telur_ayam_ras']
Regions: ['Jakarta Barat', 'Jakarta Pusat', 'Jakarta Selatan', 'Jakarta Timur', 'Jakarta Utara']


In [4]:
def get_region_from_filename(fname):
    fname_lower = fname.lower()
    for keyword, region in REGION_MAP.items():
        if keyword in fname_lower:
            return region
    return None


def parse_periode(raw_text):
    text = str(raw_text).lower().replace('periode', '').strip()
    parts = text.split()
    for i, p in enumerate(parts):
        if p in MONTH_MAP and i + 1 < len(parts):
            try:
                year = int(parts[i + 1])
                return MONTH_MAP[p], year
            except ValueError:
                continue
    return None, None


def parse_xlsx_file(fpath, region):
    df_raw = pd.read_excel(fpath, header=None)

    periode_raw = df_raw.iloc[1, 0] if len(df_raw) > 1 else ""
    bulan_num, tahun_num = parse_periode(periode_raw)
    if bulan_num is None:
        print(f"    [!] Tidak bisa baca periode: {periode_raw} | {os.path.basename(fpath)}")
        return pd.DataFrame()

    header_row_idx = None
    for i in range(len(df_raw)):
        val = str(df_raw.iloc[i, 0]).strip().lower()
        if val == 'no':
            header_row_idx = i
            break

    if header_row_idx is None:
        print(f"    [!] Baris header tidak ditemukan: {os.path.basename(fpath)}")
        return pd.DataFrame()

    header = df_raw.iloc[header_row_idx]

    day_col_map = {}
    for col_idx, val in enumerate(header):
        try:
            day = int(float(val))
            if 1 <= day <= 31:
                day_col_map[day] = col_idx
        except (ValueError, TypeError):
            pass

    records = []
    for i in range(header_row_idx + 1, len(df_raw)):
        row = df_raw.iloc[i]
        nama_raw = str(row.iloc[1]).strip().lower() if pd.notna(row.iloc[1]) else ''
        if not nama_raw or nama_raw == 'nan':
            continue

        komoditas = COMMODITY_MAP.get(nama_raw)
        if komoditas is None:
            continue

        for day, col_idx in day_col_map.items():
            try:
                val = row.iloc[col_idx]
                if pd.isna(val):
                    continue
                harga = float(val)
                if harga <= 0:
                    continue
                try:
                    tanggal = pd.Timestamp(year=tahun_num, month=bulan_num, day=day)
                except ValueError:
                    continue
                records.append({
                    'tanggal': tanggal,
                    'region': region,
                    'komoditas': komoditas,
                    'harga': harga,
                })
            except (ValueError, TypeError):
                continue

    return pd.DataFrame(records)

In [5]:
all_frames = []
files = sorted(os.listdir(DATA_DIR))
n_files = len(files)
n_parsed = 0

for fname in files:
    if not fname.endswith('.xlsx'):
        continue
    fpath = os.path.join(DATA_DIR, fname)
    region = get_region_from_filename(fname)
    if region is None:
        print(f"  [!] Tidak bisa deteksi region dari nama file: {fname}")
        continue

    df_parsed = parse_xlsx_file(fpath, region)
    if len(df_parsed) > 0:
        all_frames.append(df_parsed)
        n_parsed += 1

print(f"  File berhasil di-parse: {n_parsed}/{n_files}")

df_raw_all = pd.concat(all_frames, ignore_index=True)
print(f"  Total records mentah  : {len(df_raw_all):,}")
print(f"  Region ditemukan      : {sorted(df_raw_all['region'].unique())}")
print(f"  Komoditas ditemukan   : {sorted(df_raw_all['komoditas'].unique())}")
print(f"  Range tanggal         : {df_raw_all['tanggal'].min().date()} s.d. {df_raw_all['tanggal'].max().date()}")



  File berhasil di-parse: 120/120
  Total records mentah  : 33,324
  Region ditemukan      : ['Jakarta Barat', 'Jakarta Pusat', 'Jakarta Selatan', 'Jakarta Timur', 'Jakarta Utara']
  Komoditas ditemukan   : ['ayam_ras', 'bawang_merah', 'bawang_putih', 'beras_medium', 'beras_setra_premium', 'cabai_merah_keriting', 'cabai_rawit_merah', 'daging_sapi_murni', 'gula_pasir', 'telur_ayam_ras']
  Range tanggal         : 2024-01-01 s.d. 2025-12-31


In [6]:
df_raw_all['tanggal'] = pd.to_datetime(df_raw_all['tanggal'])
df_raw_all = df_raw_all.sort_values(['region', 'komoditas', 'tanggal']).reset_index(drop=True)

print('Tipe tanggal:', df_raw_all['tanggal'].dtype)
print('Data sudah diurutkan: region -> komoditas -> tanggal')



Tipe tanggal: datetime64[ns]
Data sudah diurutkan: region -> komoditas -> tanggal


In [7]:
df_raw_all['harga'] = pd.to_numeric(df_raw_all['harga'], errors='coerce')
n_invalid = (df_raw_all['harga'] <= 0).sum() + df_raw_all['harga'].isna().sum()

if n_invalid > 0:
    mask_invalid = (df_raw_all['harga'] <= 0) | df_raw_all['harga'].isna()
    for _, row in df_raw_all[mask_invalid].iterrows():
        log_entry(
            "prices", "harga",
            f"{row['tanggal'].date()}/{row['region']}/{row['komoditas']}",
            "nilai_tidak_valid", row['harga'], "NaN",
            "set_NaN_lalu_interpolasi", "Nilai <= 0 atau non-numeric"
        )
    df_raw_all.loc[mask_invalid, 'harga'] = np.nan
    print(f"  {n_invalid} nilai tidak valid ditemukan -> di-set NaN")
else:
    print("  Semua nilai sudah numerik dan > 0")



  Semua nilai sudah numerik dan > 0


In [8]:
KEY = ['tanggal', 'region', 'komoditas']
n_before = len(df_raw_all)
n_dup = df_raw_all.duplicated(subset=KEY).sum()

if n_dup > 0:
    print(f"  {n_dup} baris duplikat ditemukan -> ambil rata-rata harga")

    dup_sample = df_raw_all[df_raw_all.duplicated(subset=KEY, keep=False)].head(5)
    for _, row in dup_sample.iterrows():
        log_entry(
            "prices", str(KEY),
            f"{row['tanggal'].date()}/{row['region']}/{row['komoditas']}",
            "duplikasi", row['harga'], "mean",
            "group_mean", "Rata-rata semua nilai untuk kunci yang sama"
        )

    df_raw_all = df_raw_all.groupby(KEY, as_index=False)['harga'].mean().reset_index(drop=True)
    print(f"  Jumlah baris: {n_before:,} -> {len(df_raw_all):,}")
else:
    print("  Tidak ada duplikasi ditemukan")

  Tidak ada duplikasi ditemukan


In [9]:
DATE_MIN = df_raw_all['tanggal'].min()
DATE_MAX = df_raw_all['tanggal'].max()
date_range = pd.date_range(start=DATE_MIN, end=DATE_MAX, freq='D')
print(f"  Date range: {DATE_MIN.date()} s.d. {DATE_MAX.date()} ({len(date_range)} hari)")
full_idx = pd.DataFrame(
    list(product(date_range, REGIONS, TARGET_COMMODITIES)),
    columns=['tanggal', 'region', 'komoditas']
)
print(f"  Expected: {len(full_idx):,} baris ({len(date_range)} hari x {len(REGIONS)} region x {len(TARGET_COMMODITIES)} komoditas)")

df = full_idx.merge(df_raw_all, on=KEY, how='left')
print(f"  Actual  : {len(df):,} baris")

missing_summary = (
    df.groupby(['region', 'komoditas'])['harga']
    .apply(lambda x: x.isna().sum())
    .reset_index(name='n_missing')
)
print(f"\nMissing per region x komoditas (top 15):")
missing_summary.sort_values('n_missing', ascending=False).head(15)

  Date range: 2024-01-01 s.d. 2025-12-31 (731 hari)
  Expected: 36,550 baris (731 hari x 5 region x 10 komoditas)
  Actual  : 36,550 baris

Missing per region x komoditas (top 15):


,region,komoditas,n_missing
13,Jakarta Pusat,beras_medium,680
33,Jakarta Timur,beras_medium,625
43,Jakarta Utara,beras_medium,586
3,Jakarta Barat,beras_medium,584
23,Jakarta Selatan,beras_medium,431
14,Jakarta Pusat,beras_setra_premium,12
10,Jakarta Pusat,ayam_ras,9
11,Jakarta Pusat,bawang_merah,9
17,Jakarta Pusat,daging_sapi_murni,9
16,Jakarta Pusat,cabai_rawit_merah,9


In [10]:
df['harga_is_imputed'] = df['harga'].isna().astype('int8')
total_to_impute = df['harga_is_imputed'].sum()
pct_impute = total_to_impute / len(df) * 100
print(f"  Total yang akan diimputasi: {total_to_impute:,} ({pct_impute:.1f}%)")


  Total yang akan diimputasi: 3,226 (8.8%)


## Missing Values


- Gap **1-7 hari** → interpolasi linear
- Gap **8-30 hari** → interpolasi spline order 3
- Gap **> 30 hari** → bfill lalu ffill

In [11]:
def get_max_gap(series):
    max_gap = 0
    cur_gap = 0
    for v in series.isna():
        if v:
            cur_gap += 1
            max_gap = max(max_gap, cur_gap)
        else:
            cur_gap = 0
    return max_gap


impute_summary = []

for region in REGIONS:
    for komoditas in TARGET_COMMODITIES:
        mask = (df['region'] == region) & (df['komoditas'] == komoditas)
        series = df.loc[mask, 'harga'].copy()
        n_missing = series.isna().sum()

        if n_missing == 0:
            continue

        max_gap = get_max_gap(series)

        if max_gap <= 7:
            metode = "interpolasi_linear"
            series = series.interpolate(method='linear', limit_direction='both')
        elif max_gap <= 30:
            try:
                series = series.interpolate(method='spline', order=3, limit_direction='both')
                metode = "interpolasi_spline_order3"
            except Exception:
                series = series.interpolate(method='linear', limit_direction='both')
                metode = "interpolasi_linear_fallback"
        else:
            series = series.bfill().ffill()
            metode = "bfill_ffill"

        df.loc[mask, 'harga'] = series
        impute_summary.append({
            'region': region,
            'komoditas': komoditas,
            'n_missing': n_missing,
            'max_gap': max_gap,
            'metode': metode,
        })

imp_df = pd.DataFrame(impute_summary)
if len(imp_df) > 0:
    print(f"  Kombinasi yang diimputasi: {len(imp_df)}")
    display(imp_df.sort_values('max_gap', ascending=False))

remaining = df['harga'].isna().sum()
print(f"  Missing yang tersisa: {remaining}")
assert remaining == 0, "Masih ada missing value setelah imputasi!"
print("  Semua missing berhasil diimputasi")


  Kombinasi yang diimputasi: 50


,region,komoditas,n_missing,max_gap,metode
33,Jakarta Timur,beras_medium,625,569,bfill_ffill
3,Jakarta Barat,beras_medium,584,482,bfill_ffill
43,Jakarta Utara,beras_medium,586,416,bfill_ffill
13,Jakarta Pusat,beras_medium,680,414,bfill_ffill
23,Jakarta Selatan,beras_medium,431,410,bfill_ffill
41,Jakarta Utara,bawang_merah,9,5,interpolasi_linear
45,Jakarta Utara,cabai_merah_keriting,9,5,interpolasi_linear
46,Jakarta Utara,cabai_rawit_merah,9,5,interpolasi_linear
44,Jakarta Utara,beras_setra_premium,8,4,interpolasi_linear
1,Jakarta Barat,bawang_merah,6,3,interpolasi_linear


  Missing yang tersisa: 0
  Semua missing berhasil diimputasi


In [13]:
df['harga_is_outlier'] = 0

outlier_summary = []
for region in REGIONS:
    for komoditas in TARGET_COMMODITIES:
        mask = (df['region'] == region) & (df['komoditas'] == komoditas)
        series = df.loc[mask, 'harga']

        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        out_mask = (series < lower) | (series > upper)
        n_out = out_mask.sum()

        if n_out > 0:
            out_idx = series[out_mask].index
            df.loc[out_idx, 'harga_is_outlier'] = 1
            outlier_summary.append({
                'region': region,
                'komoditas': komoditas,
                'n_outlier': n_out,
                'lower': lower,
                'upper': upper,
            })

out_df = pd.DataFrame(outlier_summary)
total_out = df['harga_is_outlier'].sum()
print(f"  Total outlier di-flag: {total_out:,} ({total_out / len(df) * 100:.1f}%)")

if len(out_df) > 0:
    display(out_df.sort_values('n_outlier', ascending=False).head(20))



  Total outlier di-flag: 909 (2.5%)


,region,komoditas,n_outlier,lower,upper
27,Jakarta Timur,beras_medium,161,16500.00,16500.00
11,Jakarta Pusat,beras_medium,146,14000.00,14000.00
33,Jakarta Utara,ayam_ras,64,37500.00,49500.00
2,Jakarta Barat,beras_setra_premium,60,13437.50,16937.50
5,Jakarta Barat,daging_sapi_murni,48,133216.00,141784.00
28,Jakarta Timur,beras_setra_premium,47,14387.25,17021.25
30,Jakarta Timur,daging_sapi_murni,28,129643.50,143927.50
36,Jakarta Utara,beras_medium,26,12500.00,16500.00
39,Jakarta Utara,daging_sapi_murni,24,136000.00,152000.00
14,Jakarta Pusat,daging_sapi_murni,22,127499.50,143055.50


In [14]:
df['harga'] = df['harga'].round(2)
df['harga_is_outlier'] = df['harga_is_outlier'].astype('int8')
df['harga_is_imputed'] = df['harga_is_imputed'].astype('int8')

df = df[['tanggal', 'region', 'komoditas', 'harga', 'harga_is_outlier', 'harga_is_imputed']]

df = df.sort_values(['region', 'komoditas', 'tanggal']).reset_index(drop=True)

print(f"  Kolom : {list(df.columns)}")
print(f"  Shape : {df.shape}")
df.head(6)

  Kolom : ['tanggal', 'region', 'komoditas', 'harga', 'harga_is_outlier', 'harga_is_imputed']
  Shape : (36550, 6)


,tanggal,region,komoditas,harga,harga_is_outlier,harga_is_imputed
0,2024-01-01,Jakarta Barat,ayam_ras,38750.0,0,0
1,2024-01-02,Jakarta Barat,ayam_ras,39429.0,0,0
2,2024-01-03,Jakarta Barat,ayam_ras,39571.0,0,0
3,2024-01-04,Jakarta Barat,ayam_ras,38667.0,0,0
4,2024-01-05,Jakarta Barat,ayam_ras,40000.0,0,0
5,2024-01-06,Jakarta Barat,ayam_ras,39429.0,0,0


In [15]:
expected_rows = len(date_range) * len(REGIONS) * len(TARGET_COMMODITIES)

assert df['harga'].isna().sum() == 0, "Ada harga null!"
assert (df['harga'] > 0).all(), "Ada harga <= 0!"
assert df.duplicated(KEY).sum() == 0, "Ada duplikasi kunci!"
assert df['komoditas'].nunique() == 10, "Jumlah komoditas tidak sesuai!"
assert df['region'].nunique() == 5, "Jumlah region tidak sesuai!"
assert len(df) == expected_rows, f"Jumlah baris tidak sesuai: {len(df)} vs expected {expected_rows}"

print("Tidak ada harga null")
print("Semua harga > 0")
print("Tidak ada duplikasi (tanggal x region x komoditas)")
print("10 komoditas tersedia")
print(f"5 region tersedia: {sorted(df['region'].unique())}")
print(f"{len(df):,} baris = {len(date_range)} hari x 5 region x 10 komoditas")


Tidak ada harga null
Semua harga > 0
Tidak ada duplikasi (tanggal x region x komoditas)
10 komoditas tersedia
5 region tersedia: ['Jakarta Barat', 'Jakarta Pusat', 'Jakarta Selatan', 'Jakarta Timur', 'Jakarta Utara']
36,550 baris = 731 hari x 5 region x 10 komoditas


In [17]:
out_csv = f"{OUTPUT_DIR}/prices_daily_tidy_clean.csv"
df.to_csv(out_csv, index=False)
print(f"  -> {out_csv}  ({len(df):,} baris)")


  -> output/prices_daily_tidy_clean.csv  (36,550 baris)
